# Create Schemas and Tables

In [ ]:
%run "./initialize"

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {feature_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {feature_schema}.{staging_volume}")

## Source Tables (staging inputs for feature pipelines)

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {feature_schema}.src_customer")
spark.sql(f"""CREATE TABLE {feature_schema}.src_customer (
  CUSTOMER_ID integer,
  FIRST_NAME string,
  LAST_NAME string,
  EMAIL string,
  DELETE_FLAG boolean,
  LOAD_TIMESTAMP timestamp)
TBLPROPERTIES (delta.enableChangeDataFeed = true);""")

spark.sql(f"DROP TABLE IF EXISTS {feature_schema}.src_customer_address")
spark.sql(f"""CREATE TABLE {feature_schema}.src_customer_address (
  CUSTOMER_ID integer,
  CITY string,
  STATE string,
  LOAD_TIMESTAMP timestamp)
TBLPROPERTIES (delta.enableChangeDataFeed = true);""")

spark.sql(f"DROP TABLE IF EXISTS {feature_schema}.src_customer_purchase")
spark.sql(f"""CREATE TABLE {feature_schema}.src_customer_purchase (
  CUSTOMER_ID integer,
  PRODUCT string,
  QUANTITY integer,
  PRICE decimal(10, 2),
  PURCHASE_TIMESTAMP timestamp)
TBLPROPERTIES (delta.enableChangeDataFeed = true);""")

spark.sql(f"DROP TABLE IF EXISTS {feature_schema}.src_customer_snapshot_source")
spark.sql(f"""CREATE TABLE {feature_schema}.src_customer_snapshot_source (
  CUSTOMER_ID integer,
  FIRST_NAME string,
  LAST_NAME string,
  EMAIL string,
  DELETE_FLAG boolean,
  LOAD_TIMESTAMP timestamp)
TBLPROPERTIES (delta.enableChangeDataFeed = true);""")

spark.sql(f"DROP TABLE IF EXISTS {feature_schema}.src_customer_historical_snapshot_source")
spark.sql(f"""CREATE TABLE {feature_schema}.src_customer_historical_snapshot_source (
  CUSTOMER_ID integer,
  FIRST_NAME string,
  LAST_NAME string,
  EMAIL string,
  LOAD_TIMESTAMP timestamp)
TBLPROPERTIES (delta.enableChangeDataFeed = true);""")

spark.sql(f"DROP TABLE IF EXISTS {feature_schema}.src_customer_snapshots")
spark.sql(f"""CREATE TABLE {feature_schema}.src_customer_snapshots (
  CUSTOMER_ID integer,
  FIRST_NAME string,
  LAST_NAME string,
  EMAIL string,
  UPDATE_TIMESTAMP timestamp,
  SNAPSHOT_TIMESTAMP timestamp,
  SNAPSHOT_VERSION integer)
TBLPROPERTIES (delta.enableChangeDataFeed = true);""")

## Migration Source Tables

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {feature_schema}.src_table_to_migrate_scd0")
spark.sql(f"""CREATE TABLE {feature_schema}.src_table_to_migrate_scd0 (
  CUSTOMER_ID integer,
  FIRST_NAME string,
  LAST_NAME string,
  EMAIL string)
TBLPROPERTIES (delta.enableChangeDataFeed = true);""")

spark.sql(f"DROP TABLE IF EXISTS {feature_schema}.src_table_to_migrate_scd2")
spark.sql(f"""CREATE TABLE {feature_schema}.src_table_to_migrate_scd2 (
  CUSTOMER_ID integer,
  FIRST_NAME string,
  LAST_NAME string,
  EMAIL string,
  EFFECTIVE_FROM timestamp,
  EFFECTIVE_TO timestamp)
TBLPROPERTIES (delta.enableChangeDataFeed = true);""")

## Kafka Tables

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {feature_schema}.src_kafka_sink_sample_source")
spark.sql(f"""CREATE TABLE {feature_schema}.src_kafka_sink_sample_source (
    Message_Id BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 1 INCREMENT BY 1),
    Message_Ts TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    Message_payload STRING
)
USING delta
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.feature.allowColumnDefaults' = 'supported',
    'delta.feature.changeDataFeed' = 'supported',
    'delta.feature.columnMapping' = 'supported',
    'delta.feature.generatedColumns' = 'supported',
    'delta.feature.invariants' = 'supported',
    'delta.minReaderVersion' = '3',
    'delta.minWriterVersion' = '7'
)""")